# V5W_08 - Stessi marker? Confronto cross-coorte (coorte originale vs V5W)

Domanda (Francesco): non basta che entrambe le coorti abbiano 2 gruppi - devono essere gli STESSI gruppi, con gli stessi marker (attivazioni, bande, canali). Qui si confrontano i pattern C1-C0 (node strength + potenza per banda) tra coorte originale (tesi, eeg16b/eeg40) e V5W, canale per canale e banda per banda, + cross-proiezione.

**Env: `daniele_311`** (sul server). Usa cache eeg16b/eeg40 (originale) e v5w05/v5w07 (validazione).

## par.1 - Carica entrambe le coorti

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr, mannwhitneyu
project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
MOD = project_root/'models'; FIG_DIR = project_root/'figures'; FIG_DIR.mkdir(exist_ok=True)
N_CHAN=61; triu=np.triu_indices(N_CHAN,k=1)
def vec_to_sym(v):
    M=np.zeros((N_CHAN,N_CHAN)); M[triu]=v; return M+M.T
def ns_of(feat):                       # node strength per soggetto da triu |PCC|
    return np.array([vec_to_sym(feat[i]).sum(1)/(N_CHAN-1) for i in range(len(feat))])

# --- coorte ORIGINALE (tesi) ---
og = np.load(MOD/'eeg16b'/'feat_graph_abs_pcc.npz', allow_pickle=True)
FEAT_OG, SUBJ_OG, CHAN = og['feat'], og['subj_ids'].tolist(), list(og['chan_names'])
ogb = np.load(MOD/'eeg40'/'persubj_band_features.npz', allow_pickle=True)
POW_OG, SUBJ_OGB, BANDS = ogb['power'], ogb['subjects'].tolist(), list(ogb['bands'])   # (73,5,61)
# --- coorte V5W ---
v = np.load(MOD/'v5w05'/'feat_abs_pcc.npz', allow_pickle=True)
FEAT_V, SUBJ_V = v['feat_g'], v['subj'].tolist()
vbp = np.load(MOD/'v5w07'/'subject_bandpow.npz', allow_pickle=True)
POW_V, SUBJ_VB = vbp['bp'], vbp['subj'].tolist()    # (41,5,61)
vl = np.load(MOD/'v5w07'/'tangent_clusters.npz', allow_pickle=True)
LAB_V = {int(s):int(l) for s,l in zip(vl['subj'], vl['label'])}    # label RIEMANN V5W
print(f'OG conn: {FEAT_OG.shape}  OG band: {POW_OG.shape}  V5W conn: {FEAT_V.shape}  V5W band: {POW_V.shape}')
print(f'Bande: {BANDS}')
print('NB: si assume stesso ordine canali (montaggio ebneuro) nelle due coorti.')


## par.2 - Label k=2

In [ ]:
# Label k=2: OG ricalcolata euclidea (come tesi); V5W = label Riemann (V5W_07)
def k2(feat):
    Z=PCA(n_components=20,random_state=42).fit_transform(StandardScaler().fit_transform(feat))
    return KMeans(n_clusters=2,n_init=30,random_state=42).fit_predict(Z)
lab_og = k2(FEAT_OG)                       # (89,)
lab_v  = np.array([LAB_V[s] for s in SUBJ_V])   # (41,)
print(f'OG: C0={ (lab_og==0).sum() } C1={ (lab_og==1).sum() }   V5W: C0={ (lab_v==0).sum() } C1={ (lab_v==1).sum() }')


## par.3 - Pattern marker C1-C0 per coorte

In [ ]:
# Pattern marker C1-C0 per coorte: node strength + potenza per banda
NS_OG = ns_of(FEAT_OG); NS_V = ns_of(FEAT_V)
ns_diff_og = NS_OG[lab_og==1].mean(0) - NS_OG[lab_og==0].mean(0)     # (61,)
ns_diff_v  = NS_V[lab_v==1].mean(0)  - NS_V[lab_v==0].mean(0)

# allinea il segno (C0/C1 arbitrari): orienta V5W per correlare + con OG sulla connettivita
if pearsonr(ns_diff_og, ns_diff_v)[0] < 0:
    lab_v = 1 - lab_v; ns_diff_v = -ns_diff_v
    print('(label V5W invertite per allineare orientamento)')

# band power diff: usa solo i soggetti che hanno ANCHE la label (intersezione)
lab_og_map = {int(s): int(l) for s, l in zip(SUBJ_OG, lab_og)}
m_ogb = np.array([lab_og_map.get(int(s), -1) for s in SUBJ_OGB])
n_drop_og = int((m_ogb < 0).sum())
pdiff_og = POW_OG[m_ogb == 1].mean(0) - POW_OG[m_ogb == 0].mean(0)        # (5,61)

lab_v_map = {int(s): int(l) for s, l in zip(SUBJ_V, lab_v)}
m_vb = np.array([lab_v_map.get(int(s), -1) for s in SUBJ_VB])
n_drop_v = int((m_vb < 0).sum())
pdiff_v = POW_V[m_vb == 1].mean(0) - POW_V[m_vb == 0].mean(0)             # (5,61)
print(f'Pattern C1-C0 calcolati. Band: OG {(m_ogb>=0).sum()}/{len(SUBJ_OGB)} sogg (scartati {n_drop_og}), '
      f'V5W {(m_vb>=0).sum()}/{len(SUBJ_VB)} sogg (scartati {n_drop_v}).')


## par.4 - Stessi marker? Correlazione + cross-proiezione

In [ ]:
# CONFRONTO MARKER: gli stessi canali/bande distinguono nelle due coorti?
r_ns, p_ns = pearsonr(ns_diff_og, ns_diff_v)
print('='*62)
print('  STESSI MARKER? Correlazione cross-coorte dei pattern C1-C0')
print('='*62)
print(f'  Connettivita (node strength, 61 canali):  r = {r_ns:+.3f}  p={p_ns:.1e}')
print('  Per banda (potenza relativa, 61 canali):')
band_r=[]
for b,bn in enumerate(BANDS):
    rb,pb = pearsonr(pdiff_og[b], pdiff_v[b]); band_r.append(rb)
    flag = '  <- match' if rb>0.4 else ('  (debole)' if rb>0 else '  <- DIVERSO')
    print(f'    {bn:6s}: r = {rb:+.3f}  p={pb:.2f}{flag}')

# cross-proiezione: i soggetti V5W si separano lungo l'ASSE marker della coorte OG?
axis_og = ns_diff_og/ (np.linalg.norm(ns_diff_og)+1e-12)
proj_v = NS_V @ axis_og
u,p_proj = mannwhitneyu(proj_v[lab_v==0], proj_v[lab_v==1])
print('-'*62)
print(f'  Cross-proiezione: V5W proiettati sull asse-marker OG -> separano C0/C1?')
print(f'    Mann-Whitney p = {p_proj:.2e}  (C0={proj_v[lab_v==0].mean():+.3f} C1={proj_v[lab_v==1].mean():+.3f})')
print('='*62)

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].scatter(ns_diff_og, ns_diff_v, s=70, edgecolor='k', lw=0.5, c='#444')
axes[0].axhline(0,color='0.7',lw=.8); axes[0].axvline(0,color='0.7',lw=.8)
axes[0].set_xlabel('OG: node strength C1-C0'); axes[0].set_ylabel('V5W: node strength C1-C0')
axes[0].set_title(f'Stessa topografia di connettivita?\nr={r_ns:.3f}', fontweight='bold')
cols=['#2ca02c' if rb>0.4 else ('#ff7f0e' if rb>0 else '#d62728') for rb in band_r]
axes[1].bar(range(len(BANDS)), band_r, color=cols)
axes[1].axhline(0,color='k',lw=.8); axes[1].axhline(0.4,color='0.6',ls='--',lw=1,label='match (r>0.4)')
axes[1].set_xticks(range(len(BANDS))); axes[1].set_xticklabels(BANDS)
axes[1].set_ylabel('r cross-coorte (pattern C1-C0)'); axes[1].set_title('Stesse bande/canali?', fontweight='bold'); axes[1].legend(fontsize=8)
fig.suptitle('V5W_08 - Stessi marker tra coorte originale e V5W?', fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w08_marker_comparison.png', dpi=160, bbox_inches='tight'); plt.show()

print('\nVERDETTO:')
ok_ns = r_ns>0.5; ok_alpha = band_r[BANDS.index('alpha')]>0.4 if 'alpha' in BANDS else False
if ok_ns and ok_alpha and p_proj<0.05:
    print('  STESSI fenotipi: connettivita + alpha coincidono, e i V5W si separano')
    print('  lungo l asse-marker della coorte originale. Replica confermata (Francesco OK).')
else:
    print('  Marker NON pienamente coincidenti -> due dicotomie diverse o solo parziale match.')
print('  (bande gamma/EOG eventualmente discordanti = artefatto V5W, non il fenotipo)')
